In [1]:
import numpy as np
import torch
import torchvision.transforms as transforms
import gc
import logging
from components.other_utilities.models_to_train import ResNetPLModel
from components.FL_sim import FLSimulator
from components.other_utilities.datasets import FasterSVHN
from components.FL_sim import Agent
from components.FL_sim import FederatedModelWrapper
torch.set_float32_matmul_precision('high')

dataset = [
    FasterSVHN(
        root='../data/SVHN', split=s,
        transform=transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4377, 0.4438, 0.4728],
                std=[0.1980, 0.2010, 0.1970]
            ),
        ])
    ) for s in ['train', 'test']]

dataset = [torch.utils.data.Subset(d, list(range(100))) for d in dataset]
for i in range(10):
    for d in dataset:
        d.dataset.labels[i] = i

def pre_send_process(worker_grad_dict, agent_id):
    worker_broadcast_data = [worker_grad_dict]
    return worker_broadcast_data


def server_rec_process(worker_broadcast_data, agent_id, worker_count, global_model_dims, previous_data, ):
    # use this function to process the data received from workers
    result_dict = worker_broadcast_data[0]
    return result_dict


k = 2
batch_size = 5000
post_training_report = False

In [2]:
state_data_samples_per_batch_list_w1 = []

corr_per_batch_list = []


def compute_corr_quantile(grads_samples_1, grads_samples_2, n_quantiles=10):
    similarities = []
    for i in range(grads_samples_1.shape[1]):
        col1 = grads_samples_1[:, i]
        col2 = grads_samples_2[:, i]

        # Create quantile representations
        quantiles = torch.linspace(0, 1, n_quantiles + 1)[1:-1].to(float).to('cuda')
        q1 = torch.quantile(col1.to(float), quantiles)
        q2 = torch.quantile(col2.to(float), quantiles)

        # Pearson correlation on quantiles
        corr = torch.corrcoef(torch.stack([q1, q2]))[0, 1]
        similarities.append(corr if not torch.isnan(corr) else 0.0)

    grads_corrs = torch.tensor(similarities)
    corr_per_batch_list.append(grads_corrs.to(torch.float8_e4m3fn).cpu().numpy())


class CorrAgent(Agent):
    def train(self, train_dataloader, test_dataloader, epochs, round_s):
        assert isinstance(self.local_model, FederatedModelWrapper), \
            "Local model is not set. Call set_local_models() before training."

        self.local_model.args_for_f_on_grad['curr_round'] = round_s

        # load model to GPU memory
        self.local_model.train()
        self.local_model.to('cuda')

        self.local_model.optimizer = self.local_model.configure_optimizers()

        # train the model manually
        self.local_model.on_train_start()

        for epoch in range(epochs):
            # generate state dict per batch
            for batch_idx, (data, target) in enumerate(train_dataloader):
                self.local_model.optimizer.zero_grad()
                loss = self.local_model.training_step((data.to('cuda'), target.to('cuda')), batch_idx)
                loss.backward()

                self.local_model.on_before_optimizer_step(self.local_model.optimizer)

                self.local_model.optimizer.step()

                # record samples of grads
                samples=[]
                for _ in range(2):
                    for sample_idx, (data, target) in enumerate(train_dataloader):
                        self.local_model.optimizer.zero_grad()
                        loss = self.local_model.training_step((data.to('cuda'), target.to('cuda')), batch_idx)
                        loss.backward()

                        # store the gradients
                        grads = torch.concatenate([
                            param.grad.detach().clone().to(torch.float32).flatten()
                            for _, param in self.local_model.named_parameters()
                            if param.requires_grad])

                        samples.append(grads)

                if batch_idx!=0 and batch_idx%3!=0:
                    continue

                samples = torch.stack(samples)
                if self.agent_id == 0:
                    state_data_samples_per_batch_list_w1.append(samples)
                elif self.agent_id == 1:
                    compute_corr_quantile(state_data_samples_per_batch_list_w1.pop(0), samples)
                else: raise ValueError("Agent ID should be either 0 or 1.")

        # remove model from GPU memory
        self.local_model.eval()
        self.local_model.to('cpu')

        gc.collect()
        torch.cuda.empty_cache()

class CorrFLSimulator(FLSimulator):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.agents = [
            CorrAgent(agent_id, self.train_sampler.size_of_partition[agent_id], pre_send_process)
            for agent_id in range(len(self.agents))
        ]


In [3]:
model = ResNetPLModel(num_classes=10, resnet_version='resnet18', lr=0.01, logging_disabled=logging_disabled)

model.load_state_dict(torch.load('exp_data/resnet18_svhn.pth', map_location='cpu'))

sim = CorrFLSimulator(
    num_agents=k, communication_rounds=10, client_epochs_per_round=5,
    batch_size=batch_size, dataset_train=dataset[0], dataset_test=dataset[1],
    aggregation_method='fedavg', non_iid_sampling=False, pl_model=model,
    pre_send_process=pre_send_process, server_rec_process=server_rec_process
)

logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
sim.run_simulation(post_training_report=post_training_report, pre_training_global_epochs=0)


round 1/10 --------------------
  - reporting global model metrics
         test loss: 2.175, test auc: 0.573
         train loss: (rank='ALL') 2.154, train auc: 0.615
     > training agent 1/2
     > training agent 2/2


TypeError: Got unsupported ScalarType Float8_e4m3fn

In [ ]:
corr_per_batch_list